# 🤗 Synthetic Dataset Creation — HuggingFace Pipeline
### Real-Time Query Expansion & Topic Tagging — Stage 1

**Model:** `openai-community/gpt2` (swap to any HF chat model)  
**Platform:** Lightning AI (GPU recommended)  
**Pipeline:** Generate → Annotate → Validate  
**Saves:** CSV + JSON · Checkpoint every 50 samples · 3 retries on failure

---
**How to swap models:**  
Set `HF_MODEL_ID` in Cell 1 to any instruction-tuned chat model, e.g.:
- `"openai-community/gpt2"` — tiny, CPU-friendly (default)
- `"openai/gpt-oss-20b"` — as requested (needs GPU + HF access)
- `"mistralai/Mistral-7B-Instruct-v0.2"` — good quality, needs GPU
- `"microsoft/Phi-3-mini-4k-instruct"` — fast, small GPU
- `"meta-llama/Meta-Llama-3-8B-Instruct"` — needs HF gated access


## 0. Install Dependencies

In [1]:
!pip install transformers accelerate huggingface_hub tqdm pandas scikit-learn bitsandbytes -q
print('✅ done')

✅ done


## 1. HuggingFace Login + Config

In [ ]:
import os, json, time, re, csv, math
from pathlib import Path
from datetime import datetime
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login

# ── HF LOGIN ─────────────────────────────────────────────────────
# Option A: paste token directly (for Lightning AI secrets use option B)
HF_TOKEN = "HF_TOKEN"   # <── replace with your token

# Option B: set as env var in Lightning AI → os.environ['HF_TOKEN']
# HF_TOKEN = os.environ.get("HF_TOKEN", "")

login(token=HF_TOKEN)
print("✅ HuggingFace login successful")

# ── CONFIG ───────────────────────────────────────────────────────
# Swap this to any instruction-tuned HF model ID
HF_MODEL_ID    = "Qwen/Qwen3-14B"      # <── set your model here

TARGET_SAMPLES = 500
SAVE_EVERY     = 50
MAX_RETRIES    = 3
RETRY_DELAY    = 5
OUTPUT_DIR     = Path("dataset_output")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH        = OUTPUT_DIR / "conversations.csv"
JSON_PATH       = OUTPUT_DIR / "conversations.json"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"

ALLOWED_L1 = {"Politics","Sports","Technology","Entertainment",
              "Health","History","Geography","General"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📟 Device : {DEVICE}")
print(f"📦 Model  : {HF_MODEL_ID}")
print(f"📁 Output : {OUTPUT_DIR.resolve()}")

✅ HuggingFace login successful
📟 Device : cuda
📦 Model  : Qwen/Qwen3-14B
📁 Output : /teamspace/studios/this_studio/dataset_output


## 2. Load Model & Tokenizer

In [3]:
print(f"Loading {HF_MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(
    HF_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Set pad token if missing (common for GPT-style models)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    # Uncomment for 4-bit quantization on smaller GPUs:
    # load_in_4bit=True,
)

if DEVICE == "cpu":
    model = model.to(DEVICE)

model.eval()
print(f"✅ Model loaded on {DEVICE}")

# ── CHECK IF MODEL SUPPORTS CHAT TEMPLATE ───────────────────────
HAS_CHAT_TEMPLATE = hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None
print(f"   Chat template : {'yes' if HAS_CHAT_TEMPLATE else 'no — will use prompt injection fallback'}")

Loading Qwen/Qwen3-14B ...


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Model loaded on cuda
   Chat template : yes


## 3. LLM Caller

In [ ]:
def call_llm(system_prompt, user_prompt, temperature=0.8, max_new_tokens=2500):
    """
    Calls the local HF model with retry logic.
    Handles Chat Templates robustly and strips DeepSeek R1 reasoning tags.
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            if HAS_CHAT_TEMPLATE:
                # 1. Get the prompt as a plain string first (safest method)
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ]
                prompt_str = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False  # <--- It goes right here!
                )
            else:
                # Fallback
                prompt_str = (
                    f"### System:\n{system_prompt}\n\n"
                    f"### User:\n{user_prompt}\n\n"
                    f"### Response:\n"
                )

            # 2. Tokenize the string (this perfectly generates input_ids and attention_mask)
            inputs = tokenizer(prompt_str, return_tensors="pt").to(DEVICE)
            prompt_len = inputs.input_ids.shape[-1]

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,  # Unpacking passes both input_ids and attention_mask safely
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=temperature > 0,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # 3. Decode only the new tokens
            new_tokens = output_ids[0][prompt_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)

            # 4. DeepSeek R1 Fix: Strip out the <think>...</think> reasoning blocks
            import re
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

            return text

        except Exception as e:
            last_err = e
            print(f"  ⚠️  Attempt {attempt}/{MAX_RETRIES}: {type(e).__name__}: {e}")
            if attempt < MAX_RETRIES:
                print(f"  ⏳  Retry in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY)

    raise RuntimeError(f"All {MAX_RETRIES} retries failed. Last: {last_err}")

print("✅ call_llm() ready.")

✅ call_llm() ready.


## 4. Seed Pairs

In [5]:
BASE_SEEDS = [
    ("Politics","India","Narendra Modi"),("Politics","India","Rahul Gandhi"),
    ("Politics","India","Manmohan Singh"),("Politics","UK","Rishi Sunak"),
    ("Politics","UK","Boris Johnson"),("Politics","USA","Joe Biden"),
    ("Politics","USA","Donald Trump"),("Politics","USA","Barack Obama"),
    ("Sports","Cricket","Virat Kohli"),("Sports","Cricket","MS Dhoni"),
    ("Sports","Cricket","Rohit Sharma"),("Sports","Football","Lionel Messi"),
    ("Sports","Football","Cristiano Ronaldo"),("Sports","Tennis","Novak Djokovic"),
    ("Sports","Tennis","Rafael Nadal"),("Sports","Olympics","Neeraj Chopra"),
    ("Technology","AI","OpenAI"),("Technology","AI","Geoffrey Hinton"),
    ("Technology","AI","Google DeepMind"),("Technology","Space","ISRO"),
    ("Technology","Space","SpaceX"),("Technology","Gadgets","Apple iPhone"),
    ("Entertainment","Bollywood","Shah Rukh Khan"),("Entertainment","Bollywood","Aamir Khan"),
    ("Entertainment","Hollywood","Christopher Nolan"),("Entertainment","Hollywood","Marvel Studios"),
    ("Health","Nutrition","intermittent fasting"),("Health","Mental Health","anxiety"),
    ("Health","Fitness","yoga"),("History","India","Mahatma Gandhi"),
    ("History","World Wars","World War II"),("Geography","Countries","India"),
    ("Geography","Cities","Paris"),("General","General",""),
]

repeats   = math.ceil(TARGET_SAMPLES / len(BASE_SEEDS))
ALL_SEEDS = (BASE_SEEDS * repeats)[:TARGET_SAMPLES]
print(f"Base seeds: {len(BASE_SEEDS)} | Expanded: {len(ALL_SEEDS)}")
print("\nLabel distribution:")
for k, v in sorted(Counter(s[0] for s in ALL_SEEDS).items()):
    print(f"  {k:<18} {v:>4}")

Base seeds: 34 | Expanded: 500

Label distribution:
  Entertainment        58
  General              14
  Geography            28
  Health               42
  History              28
  Politics            120
  Sports              120
  Technology           90


## 5. Prompts (Compact)

In [6]:
# ── STAGE 1: CONVERSATION GENERATION ────────────────────────────
STAGE1_SYSTEM = """Generate a realistic 24-message chat (12 User + 12 Assistant, strictly alternating, User first) for NLP training.
Rules:
- Stay on the given TOPIC and ENTITY throughout.
- User messages must mix: direct questions, pronoun refs (his/her/their), elliptical follow-ups (what about X?), at least 1 interruption (brb/wait), resumption after break, a topic switch, and a comparison.
- Assistant replies: factual, concise, contextual.
- Output ONLY the conversation. No labels or extra text.
Format:
User: ...
Assistant: ...
(repeat, exactly 24 lines)

Example (Politics-India / Narendra Modi):
User: who is the pm of india?
Assistant: Narendra Modi has been Prime Minister since 2014.
User: what are his duties?
Assistant: He leads the Cabinet, sets policy, and represents India globally.
User: and internationally?
Assistant: He attends G20, UN summits and handles bilateral diplomacy.
User: wait a sec, brb
Assistant: Sure!
User: back — what about before him?
Assistant: Manmohan Singh was PM from 2004–2014.
User: same responsibilities?
Assistant: Yes, the PM's constitutional role stays the same.
User: who leads opposition?
Assistant: Rahul Gandhi leads the main opposition, the INC.
User: compare both
Assistant: Modi (BJP) focuses on growth; Rahul (INC) emphasises welfare.
User: got it thanks
Assistant: Happy to help!"""

def stage1_user_prompt(l1, l2, entity):
    e = entity or "general small talk"
    return f"Topic: {l1} - {l2}\nEntity: {e}\n\nGenerate the 20-message conversation now."


# ── STAGE 2: ANNOTATION ──────────────────────────────────────────
STAGE2_SYSTEM = """You are an NLP annotator. For EVERY User message in the conversation, output one JSON object.
Fields:
  turn (int, odd: 1,3,5...19), raw (exact text), expanded (self-contained rewrite resolving all pronouns/ellipsis using context; copy as-is if small-talk), topic_l1 (one of: Politics/Sports/Technology/Entertainment/Health/History/Geography/General), topic_l2 (sub-topic), named_entities (list), needs_expansion (bool)
Rules:
- Output ONLY a valid JSON array of exactly 10 objects. No markdown, no extra text.
- expanded must never be empty.
- named_entities must always be a list.
Example:
[{"turn":1,"raw":"who is the pm?","expanded":"who is the prime minister of india?","topic_l1":"Politics","topic_l2":"India","named_entities":["India"],"needs_expansion":false},
 {"turn":3,"raw":"what are his duties?","expanded":"what are Narendra Modi's duties as PM of India?","topic_l1":"Politics","topic_l2":"India","named_entities":["Narendra Modi","India"],"needs_expansion":true}]"""

def stage2_user_prompt(conv_text):
    return f"Annotate every User message. Return ONLY the JSON array.\n\n{conv_text}"


# ── STAGE 3: QUALITY CHECK ───────────────────────────────────────
STAGE3_SYSTEM = "Answer with exactly one word: VALID or INVALID."

def stage3_user_prompt(raw, expanded, l1, l2):
    return f"Raw: {raw}\nExpanded: {expanded}\nTopic: {l1}>{l2}\nIs expansion faithful and topic correct?"

print("✅ Prompts loaded.")

✅ Prompts loaded.


## 6. Helpers (identical to Groq pipeline)

In [7]:
def parse_conversation(raw):
    turns = []
    for line in raw.strip().split("\n"):
        line = line.strip()
        if line.lower().startswith("user:"):
            turns.append({"role":"user",      "text":line[5:].strip()})
        elif line.lower().startswith("assistant:"):
            turns.append({"role":"assistant", "text":line[10:].strip()})
    return turns if len(turns) == 20 else None


def parse_annotation(raw_json):
    text = re.sub(r'^```[a-z]*\s*','',raw_json.strip())
    text = re.sub(r'\s*```$','',text)
    try:
        data = json.loads(text)
        return data if isinstance(data, list) else None
    except json.JSONDecodeError:
        return None


def validate_sample(turns, annotation):
    if len(turns) != 20:
        return False, f"turn count={len(turns)}"
    expected = ["user" if i%2==0 else "assistant" for i in range(20)]
    if [t["role"] for t in turns] != expected:
        return False, "alternation violated"
    if not isinstance(annotation, list) or len(annotation) != 10:
        return False, f"annotation length={len(annotation) if annotation else 'None'}"
    required = {"turn","raw","expanded","topic_l1","topic_l2","named_entities","needs_expansion"}
    for i, ann in enumerate(annotation):
        if required - set(ann.keys()):
            return False, f"ann[{i}] missing fields"
        if not str(ann.get("expanded","")).strip():
            return False, f"ann[{i}] empty expanded"
        if not isinstance(ann.get("named_entities"), list):
            return False, f"ann[{i}] entities not list"
        if ann.get("topic_l1") not in ALLOWED_L1:
            return False, f"ann[{i}] bad topic_l1='{ann.get('topic_l1')}'"
    return True, "OK"


def build_row(conv_id, l1, l2, entity, turns, annotation):
    return {
        "conversation_id":  conv_id,
        "topic_l1":         l1,
        "topic_l2":         l2,
        "seed_entity":      entity,
        "turns":            json.dumps(turns, ensure_ascii=False),
        "user_queries":     json.dumps([a["raw"]            for a in annotation], ensure_ascii=False),
        "expanded_queries": json.dumps([a["expanded"]        for a in annotation], ensure_ascii=False),
        "named_entities":   json.dumps([a["named_entities"]  for a in annotation], ensure_ascii=False),
        "needs_expansion":  json.dumps([a["needs_expansion"] for a in annotation]),
        "created_at":       datetime.utcnow().isoformat(),
    }


CSV_FIELDS = ["conversation_id","topic_l1","topic_l2","seed_entity",
              "turns","user_queries","expanded_queries",
              "named_entities","needs_expansion","created_at"]

def append_to_csv(row):
    write_header = not CSV_PATH.exists()
    with open(CSV_PATH,"a",newline="",encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header: w.writeheader()
        w.writerow(row)

def flush_to_json(buffer):
    existing = []
    if JSON_PATH.exists():
        with open(JSON_PATH) as f:
            try: existing = json.load(f)
            except: pass
    parsed = []
    for row in buffer:
        r = dict(row)
        for field in ["turns","user_queries","expanded_queries","named_entities","needs_expansion"]:
            try: r[field] = json.loads(r[field])
            except: pass
        parsed.append(r)
    with open(JSON_PATH,"w",encoding="utf-8") as f:
        json.dump(existing + parsed, f, indent=2, ensure_ascii=False)

def save_checkpoint(ids):
    with open(CHECKPOINT_PATH,"w") as f:
        json.dump({"completed":list(ids),"saved_at":datetime.utcnow().isoformat()}, f)

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH) as f:
            return set(json.load(f).get("completed",[]))
    return set()

print("✅ Helpers ready.")

✅ Helpers ready.


## 7. Main Generation Loop
```
seed → Stage1 (generate) → Stage2 (annotate) → Stage3 (validate) → save
                                                                  → checkpoint every 50
```

In [9]:
completed_ids = load_checkpoint()
print(f"📌 Checkpoint: {len(completed_ids)} samples already done.")

stats  = {"attempted":0,"stage1_failures":0,"stage2_failures":0,
          "validation_fails":0,"saved":len(completed_ids)}
buffer = []

pbar = tqdm(total=TARGET_SAMPLES, initial=len(completed_ids),
            desc="Generating", unit="sample")

for idx, (l1, l2, entity) in enumerate(ALL_SEEDS):

    conv_id = f"conv_{idx:04d}_{l1.lower()}_{l2.lower()}".replace(" ","_")

    if conv_id in completed_ids:
        continue
    if stats["saved"] >= TARGET_SAMPLES:
        tqdm.write(f"\n🎯 Target {TARGET_SAMPLES} reached.")
        break

    stats["attempted"] += 1

    # ── STAGE 1 ───────────────────────────────────────────────────
    try:
        raw_conv = call_llm(STAGE1_SYSTEM, stage1_user_prompt(l1,l2,entity),
                            temperature=0.85, max_new_tokens=1500)
    except RuntimeError as e:
        stats["stage1_failures"] += 1
        tqdm.write(f"❌ [{conv_id}] Stage1 failed: {e}")
        continue

    turns = parse_conversation(raw_conv)
    if turns is None:
        stats["stage1_failures"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Stage1 parse error.")
        tqdm.write(f"--- RAW (first 400 chars) ---\n{raw_conv[:400]}\n---")
        continue

    conv_text = "\n".join(
        f"{'User' if t['role']=='user' else 'Assistant'}: {t['text']}"
        for t in turns
    )

    # ── STAGE 2 ───────────────────────────────────────────────────
    try:
        raw_ann = call_llm(STAGE2_SYSTEM, stage2_user_prompt(conv_text),
                           temperature=0.2, max_new_tokens=2500)
    except RuntimeError as e:
        stats["stage2_failures"] += 1
        tqdm.write(f"❌ [{conv_id}] Stage2 failed: {e}")
        continue

    annotation = parse_annotation(raw_ann)
    if annotation is None:
        stats["stage2_failures"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Stage2 JSON parse failed.")
        continue

    # ── STAGE 3 ───────────────────────────────────────────────────
    is_valid, reason = validate_sample(turns, annotation)
    if not is_valid:
        stats["validation_fails"] += 1
        tqdm.write(f"⚠️  [{conv_id}] Validation failed: {reason}.")
        continue

    # ── SAVE ──────────────────────────────────────────────────────
    row = build_row(conv_id, l1, l2, entity, turns, annotation)
    append_to_csv(row)
    buffer.append(row)
    completed_ids.add(conv_id)
    stats["saved"] += 1
    pbar.update(1)

    # ── CHECKPOINT EVERY 50 ───────────────────────────────────────
    if stats["saved"] % SAVE_EVERY == 0:
        flush_to_json(buffer)
        save_checkpoint(completed_ids)
        buffer = []
        tqdm.write(f"💾 Checkpoint @ {stats['saved']} samples.")

    # No API rate limit here — but yield CPU/GPU between iterations
    time.sleep(0.1)

pbar.close()

if buffer:
    flush_to_json(buffer)
    save_checkpoint(completed_ids)
    print(f"💾 Final flush: {len(buffer)} samples.")

print("\n" + "="*50)
print("✅ Done!")
print(f"   Saved             : {stats['saved']}")
print(f"   Stage1 failures   : {stats['stage1_failures']}")
print(f"   Stage2 failures   : {stats['stage2_failures']}")
print(f"   Validation fails  : {stats['validation_fails']}")
print(f"   CSV  → {CSV_PATH}")
print(f"   JSON → {JSON_PATH}")

📌 Checkpoint: 0 samples already done.


Generating:   0%|          | 0/500 [00:00<?, ?sample/s]

⚠️  [conv_0000_politics_india] Stage1 parse error.
--- RAW (first 400 chars) ---
User: who is the pm of india?  
Assistant: Narendra Modi has been Prime Minister since 2014.  
User: what are his duties?  
Assistant: He leads the Cabinet, sets policy, and represents India globally.  
User: and internationally?  
Assistant: He attends G20, UN summits and handles bilateral diplomacy.  
User: wait a sec, brb  
Assistant: Sure!  
User: back — what about before him?  
Assistant: Man
---
⚠️  [conv_0001_politics_india] Stage1 parse error.
--- RAW (first 400 chars) ---
User: who is rahul gandhi?
Assistant: Rahul Gandhi is the leader of the Indian National Congress and a prominent opposition figure.
User: what are his main responsibilities?
Assistant: He leads the Congress party, participates in parliamentary debates, and represents the opposition in the Lok Sabha.
User: and his political background?
Assistant: He is the son of former Prime Minister Rajiv Gandhi 
---


## 8. Inspect & Verify

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows : {len(df)}")
print("\n── topic_l1 ──")
print(df["topic_l1"].value_counts().to_string())
print("\n── topic_l2 (top 15) ──")
print(df["topic_l2"].value_counts().head(15).to_string())

In [ ]:
sample = df.iloc[0]
print(f"ID     : {sample['conversation_id']}")
print(f"Topic  : {sample['topic_l1']} > {sample['topic_l2']}")
print(f"Entity : {sample['seed_entity']}\n")
print("── CONVERSATION ─────────────────────────────")
for t in json.loads(sample["turns"]):
    role = "User      " if t["role"]=="user" else "Assistant "
    print(f"  {role}: {t['text']}")
print("\n── ANNOTATIONS ──────────────────────────────")
for i,(q,e,ents,ne) in enumerate(zip(
    json.loads(sample["user_queries"]),
    json.loads(sample["expanded_queries"]),
    json.loads(sample["named_entities"]),
    json.loads(sample["needs_expansion"]))):
    print(f"  Turn {2*i+1}:")
    print(f"    Raw      : {q}")
    print(f"    Expanded : {e}")
    print(f"    Entities : {ents}")
    print(f"    Expanded?: {ne}\n")

In [ ]:
all_flags = []
for row in df["needs_expansion"]:
    all_flags.extend(json.loads(row))
total = len(all_flags)
exp   = sum(all_flags)
print(f"Total user turns       : {total}")
print(f"Needed expansion       : {exp:>5}  ({100*exp/total:.1f}%)")
print(f"Already self-contained : {total-exp:>5}  ({100*(total-exp)/total:.1f}%)")

## 9. Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["topic_l1"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["topic_l1"])

train_df.to_csv(OUTPUT_DIR/"train.csv", index=False)
val_df.to_csv(  OUTPUT_DIR/"val.csv",   index=False)
test_df.to_csv( OUTPUT_DIR/"test.csv",  index=False)

print(f"Train : {len(train_df):>4} rows (80%)")
print(f"Val   : {len(val_df):>4} rows (10%)")
print(f"Test  : {len(test_df):>4} rows (10%)")
print(f"\nFiles saved to {OUTPUT_DIR}/")